# Learning 01: LCEL Basics

LangChain Expression Language (LCEL) lets you compose chains with the `|` pipe operator. Every component is a `Runnable` with `.invoke()`, `.stream()`, and `.batch()`.

## Setup

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

OPENAI_API_KEY = "your-api-key-here"
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
print("✓ LLM initialized")


## 1. Your First Chain: Ticket Classifier

A chain is: **prompt → LLM → parser**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", 'Classify the support ticket. Return ONLY valid JSON with keys "category" (billing/technical/account/shipping) and "priority" (high/medium/low). No markdown, no explanation.'),
    ("human", "{ticket}"),
])

chain = prompt | llm | JsonOutputParser()

result = chain.invoke({"ticket": "I was charged $49.99 twice this billing cycle"})
print(result)
# → {'category': 'billing', 'priority': 'high'}


## 2. Single Call, Batch, and Streaming

All three modes use the same chain — only the calling method changes.

In [ ]:
# Single call
result = chain.invoke({"ticket": "API returns 500 errors in production"})
print("Single:", result)

# Batch (parallel calls — much faster than a loop)
tickets_batch = [
    {"ticket": "I cannot reset my password"},
    {"ticket": "My package has not arrived after 2 weeks"},
    {"ticket": "Dark mode does not persist between sessions"},
]
results = chain.batch(tickets_batch)
for t, r in zip(tickets_batch, results):
    print(f"  {t['ticket'][:40]!r:45} → {r}")


In [ ]:
# Streaming — tokens arrive as they are generated
from langchain_core.output_parsers import StrOutputParser

str_chain = prompt | llm | StrOutputParser()

print("Streaming: ", end="")
for chunk in str_chain.stream({"ticket": "My account was hacked"}):
    print(chunk, end="", flush=True)
print()


## 3. `RunnablePassthrough` — keep the original input

When you need both the original input AND some derived value in the same prompt:

In [ ]:
from langchain_core.runnables import RunnablePassthrough

context_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the support question using the provided context. Be concise."),
    ("human", "Context: {context}\n\nQuestion: {question}"),
])

# We'll build a real RAG chain in the next notebook
# For now just see how RunnablePassthrough threads values through
print("RunnablePassthrough lets you keep the original input alongside transformed values")
print("Usage: {context: retriever | format_docs, question: RunnablePassthrough()}")


## 4. `RunnableLambda` — wrap any Python function

In [ ]:
from langchain_core.runnables import RunnableLambda

# Preprocess input before sending to LLM
preprocess = RunnableLambda(lambda x: {"ticket": x.strip().lower()})
preprocess_chain = preprocess | chain

result = preprocess_chain.invoke("  I WAS CHARGED TWICE  ")
print("Preprocessed result:", result)


## Summary

- LCEL uses `|` to compose `Runnable` components
- Every chain supports `.invoke()`, `.stream()`, `.batch()`
- `RunnablePassthrough` threads the input unchanged
- `RunnableLambda` wraps any Python function
- The same chain works for single calls, streaming, and batch